# 3. Modelos benchmark y mejores combinaciones

Los 7 modelos benchmark de regresión (KNN, Ridge, Lasso, árbol de decisión, Random Forest, XGBoost y SVR) se
optimizaron con 4 métodos (Grid, Random, Bayesiano TPE y Genético) dentro de una validación anidada temporal
(secciones 3.1 a 3.11, del Entregable 2). Este capítulo documenta la selección de la mejor combinación de cada modelo,
su reentrenamiento final y su serialización para el dashboard.

In [1]:
# Localiza la raíz del proyecto (la carpeta con src/ y app.py) para poder importar los módulos de src/
import sys
from pathlib import Path
raiz = Path.cwd().resolve()
while not ((raiz / "src").is_dir() and (raiz / "app.py").exists()):
    raiz = raiz.parent
sys.path.insert(0, str(raiz))
# Librerías de análisis y módulos propios: configuración, datos, EDA, evaluación y utilidades del libro
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from src import config as cfg, datos, eda, evaluacion as ev
from src.libro import md, tabla, mostrar_plotly
from src.eda import es, p_fmt
# Estilo común de los gráficos, semilla global (SEMILLA = 42) y formato de las tablas
eda.configurar_graficos()
cfg.fijar_semillas()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

In [2]:
import json
from src import artefactos as A, data as D
L = D.lideres()
tabla(L[["modelo", "optimizador", "RMSLE bucle externo (media ± d.e.)", "RMSLE validación < 2016", "RMSLE origen móvil 2021–2026",
         "mejor optimizador por bucle externo", "inferencia (ms/fila)"]].round(4))

,modelo,optimizador,RMSLE bucle externo (media ± d.e.),RMSLE validación < 2016,RMSLE origen móvil 2021–2026,mejor optimizador por bucle externo,inferencia (ms/fila)
0,Random Forest,Genético (DEAP),0.3192 ± 0.0256,"0,386","0,328",Random,"0,013"
1,XGBoost,Bayesiano (TPE),0.3182 ± 0.0266,"0,387","0,327",Random,"0,006"
2,KNN,Bayesiano (TPE),0.3239 ± 0.0241,"0,394","0,335",Random,"0,145"
3,SVR,Genético (DEAP),0.3641 ± 0.0662,"0,395","0,355",Bayesiano (TPE),"0,094"
4,Árbol de decisión,Bayesiano (TPE),0.3311 ± 0.0315,"0,397","0,331",Grid,"0,004"
5,Ridge,Grid,0.3341 ± 0.0251,"0,401","0,346",Genético (DEAP),"0,003"
6,Lasso,Grid,0.3341 ± 0.0251,"0,401","0,346",Genético (DEAP),"0,006"
7,A+B v2 (propio),rejilla interna de λ,— (evaluado en el origen móvil),—,"0,326",—,"0,314"


## Criterio de selección

- **Mejor combinación de cada modelo.** En cada familia se elige la combinación con menor RMSLE en la validación
  interna con datos anteriores a 2016, la misma que fija sus hiperparámetros para el origen móvil. Así, ningún mes de
  2021–2026 interviene en la elección, y los modelos del dashboard son exactamente los que se compararon
  estadísticamente (capítulo 6). La columna «mejor optimizador por bucle externo» muestra la combinación que habría
  ganado con la media del bucle externo de la validación anidada; las diferencias entre optimizadores de una misma
  familia son menores que la variabilidad entre pliegues (sección 3.8).
- **Mejor combinación global.** A+B v2 tiene el menor RMSLE en el origen móvil y es el único modelo en el Model
  Confidence Set (capítulo 6). Entre los benchmarks, el campeón de XGBoost es el mejor y quedó fuera del MCS por poco.

## Reentrenamiento final y serialización

Cada pipeline completo (preprocesamiento + modelo, con el objetivo en escala log1p) se reentrena con **todos los meses
con objetivo conocido** y se guarda con `joblib.dump(..., compress=3)` en `artifacts/`, junto con un `metadata.json`
(versiones de librerías, semilla, hiperparámetros, esquema de columnas con tipos y rangos válidos y métricas). El
dashboard solo carga estos archivos (`scripts/entrenar_artefactos.py`).

**Prueba de ida y vuelta.** Antes de guardar, cada modelo predice los últimos 12 meses del entrenamiento; el archivo se
carga de nuevo y debe producir exactamente las mismas predicciones. Las predicciones de referencia quedan en
`artifacts/*.ida_vuelta.npy` y `tests/test_artefactos.py` repite la verificación. En otra máquina, el álgebra lineal
puede cambiar el último dígito de los modelos lineales y de A+B (diferencias del orden de 1e-14), por eso la prueba
unitaria compara con un error relativo máximo de 1e-9.

In [3]:
filas = []
for etq in D.modelos():
    m = A.metadata(etq)
    filas.append({"modelo": etq, "archivo": m["archivo"], "tamaño (MB)": m["tamano_MB"],
                  "filas de entrenamiento": m["entrenamiento"]["filas"], "último mes objetivo": m["entrenamiento"]["ultimo_mes_objetivo"],
                  "ida y vuelta idéntica": "sí" if m["prueba_ida_y_vuelta"]["identicas"] else "NO",
                  "carga (s)": m["tiempo_carga_s"], "inferencia (ms/fila)": m["tiempo_inferencia_ms_por_fila"]})
tabla(pd.DataFrame(filas))

,modelo,archivo,tamaño (MB),filas de entrenamiento,último mes objetivo,ida y vuelta idéntica,carga (s),inferencia (ms/fila)
0,A+B v2,A+B_v2.joblib,"3,150",42.380,2026-09-01,sí,"0,328","0,314"
1,Random Forest · AG,Random_Forest__AG.joblib,"0,730",42.380,2026-09-01,sí,"0,075","0,013"
2,XGBoost · TPE,XGBoost__TPE.joblib,"0,150",42.380,2026-09-01,sí,"0,014","0,006"
3,KNN · TPE,KNN__TPE.joblib,"0,790",42.380,2026-09-01,sí,"0,052","0,145"
4,SVR · AG,SVR__AG.joblib,"0,070",42.380,2026-09-01,sí,"0,008","0,094"
5,Árbol de decisión · TPE,Arbol_de_decision__TPE.joblib,"0,010",42.380,2026-09-01,sí,"0,003","0,004"
6,Ridge · Grid,Ridge__Grid.joblib,"0,000",42.380,2026-09-01,sí,"0,003","0,003"
7,Lasso · Grid,Lasso__Grid.joblib,"0,000",42.380,2026-09-01,sí,"0,003","0,006"
